In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

HADOOP_HOME: C:\hadoop


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Olist_ETL")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)

c:\Users\girdh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [3]:
raw_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\raw"

In [4]:
orders = spark.read.csv(
    f"{raw_path}/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

In [5]:
orders.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [6]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [7]:
order_items = spark.read.csv(
    f"{raw_path}/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

In [8]:
order_items.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [9]:
order_items.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [10]:
products = spark.read.csv(
    f"{raw_path}/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

In [11]:
products.show(5)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [12]:
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [13]:
sellers = spark.read.csv(
    f"{raw_path}/olist_sellers_dataset.csv",
    header=True,
    inferSchema=True
)
sellers.show(5)

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows


In [14]:
sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [15]:
customers = spark.read.csv(
    f"{raw_path}/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)
customers.show(5)
customers.printSchema()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows
root
 |-- customer_id: string (

In [16]:
payments = spark.read.csv(
    f"{raw_path}/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

payments.show(5)
payments.printSchema()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments:

In [17]:
reviews = spark.read.csv(
    f"{raw_path}/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)
reviews.show(5)
reviews.printSchema()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [18]:
geolocation = spark.read.csv(
    f"{raw_path}/olist_geolocation_dataset.csv",
    header=True,
    inferSchema=True
)

In [19]:
geolocation.show(5)
geolocation.printSchema()

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
|                       1046| -23.54612896641469|-46.64295148361138|       sao paulo|               SP|
|                       1041|  -23.5443921648681|-46.63949930627844|       sao paulo|               SP|
|                       1035|-23.541577961711493|-46.64160722329613|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 5 rows
root
 |-- geolocation_zip_code_prefix: i

In [20]:
category_translation = spark.read.csv(
    f"{raw_path}/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

category_translation.show(5)
category_translation.printSchema()

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
+---------------------+-----------------------------+
only showing top 5 rows
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [21]:
from pyspark.sql.functions import col

reviews = reviews.withColumn(
    "review_score",
    col("review_score").cast("integer")
)

In [22]:
reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [23]:
from pyspark.sql.functions import col

reviews = (
    reviews
    .withColumn("review_creation_date", col("review_creation_date").cast("timestamp"))
    .withColumn("review_answer_timestamp", col("review_answer_timestamp").cast("timestamp"))
)

In [24]:
reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [25]:
reviews = reviews.select("order_id", "review_score")

In [26]:
from pyspark.sql.functions import col

order_items = order_items.withColumn(
    "item_total",
    col("price") + col("freight_value")
)

In [27]:
order_items.select("item_total").show(5)

+------------------+
|        item_total|
+------------------+
|             72.19|
|            259.83|
|            216.87|
|             25.78|
|218.04000000000002|
+------------------+
only showing top 5 rows


In [28]:
order_items = order_items.join(
    orders,
    on="order_id",
    how="left"
)

In [29]:
order_items.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|        item_total|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017

In [30]:
order_items = order_items.join(
    customers,
    on="customer_id",
    how="left"
)

In [31]:
print(order_items.count(), len(order_items.columns))

112650 19


In [32]:
order_items = order_items.join(
    products,
    on="product_id",
    how="left"
)

In [33]:
print(order_items.count(), len(order_items.columns))

112650 27


In [34]:
order_items = order_items.join(
    sellers,
    on="seller_id",
    how="left"
)

In [35]:
print(order_items.count(), len(order_items.columns))

112650 30


In [36]:
from pyspark.sql.functions import sum

payment_summary = payments.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value")
)

In [37]:
payment_summary.show(5)

+--------------------+-------------------+
|            order_id|total_payment_value|
+--------------------+-------------------+
|bb2d7e3141540afc2...|              37.15|
|85be7c94bcd3f908f...|              72.75|
|8ca5bdac5ebe8f2d6...| 189.07999999999998|
|54066aeaaf3ac32e7...|             148.06|
|5db54d41d5ebd6d76...|             136.26|
+--------------------+-------------------+
only showing top 5 rows


In [38]:
from pyspark.sql.functions import collect_set, concat_ws

payment_types = payments.groupBy("order_id").agg(
    concat_ws(", ", collect_set("payment_type")).alias("payment_types")
)

In [39]:
payment_types.show(5)

+--------------------+-------------+
|            order_id|payment_types|
+--------------------+-------------+
|00018f77f2f0320c5...|  credit_card|
|00042b26cf59d7ce6...|  credit_card|
|00054e8431b9d7675...|  credit_card|
|0006ec9db01a64e59...|  credit_card|
|000aed2e25dbad2f9...|  credit_card|
+--------------------+-------------+
only showing top 5 rows


In [40]:
payment_types.orderBy("order_id", ascending=False).show(5)

+--------------------+-------------+
|            order_id|payment_types|
+--------------------+-------------+
|fffe41c64501cc87c...|  credit_card|
|fffe18544ffabc95d...|  credit_card|
|fffce4705a9662cd7...|  credit_card|
|fffcd46ef2263f404...|       boleto|
|fffc94f6ce00a0058...|       boleto|
+--------------------+-------------+
only showing top 5 rows


In [41]:
from pyspark.sql.functions import max

payment_installments = payments.groupBy("order_id").agg(
    max("payment_installments").alias("max_payment_installments")
)

In [42]:
payment_installments.show(5)

+--------------------+------------------------+
|            order_id|max_payment_installments|
+--------------------+------------------------+
|bb2d7e3141540afc2...|                       1|
|85be7c94bcd3f908f...|                       1|
|8ca5bdac5ebe8f2d6...|                       2|
|54066aeaaf3ac32e7...|                       2|
|5db54d41d5ebd6d76...|                       2|
+--------------------+------------------------+
only showing top 5 rows


In [43]:
from pyspark.sql.functions import sum, collect_set, concat_ws, max, count

payment_summary = payments.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value"),
    concat_ws(", ", collect_set("payment_type")).alias("payment_types"),
    max("payment_installments").alias("max_payment_installments"),
    count("*").alias("payment_count")
)

In [44]:
order_items = order_items.join(
    payment_summary,
    on="order_id",
    how="left"
)

In [45]:
print(order_items.count(), len(order_items.columns))

112650 34


In [46]:
reviews_raw = spark.read.csv(
    f"{raw_path}/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False
)

reviews_raw.select(
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(10, truncate=False)

+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                        |review_creation_date|review_answer_timestamp|
+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                                             

In [47]:
reviews_raw.filter(
    ~col("review_score").isin("1", "2", "3", "4", "5")
).select(
    "order_id",
    "review_score"
).show(20, truncate=False)

+-------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+
|order_id                                                                                                     |review_score                                                     |
+-------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+
|2018-03-23 00:00:00                                                                                          |2018-04-01 00:27:51                                              |
|2018-01-28 00:00:00                                                                                          |2018-02-04 11:06:15                                              |
|2017-06-24 00:00:00                                                                                          

In [48]:
reviews_raw = spark.read.csv(
    f"{raw_path}/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False,
    multiLine=True,
    quote='"',
    escape='"'
)

reviews_raw.select(
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(10, truncate=False)

+--------------------------------+------------+--------------------+-----------------------+
|order_id                        |review_score|review_creation_date|review_answer_timestamp|
+--------------------------------+------------+--------------------+-----------------------+
|73fc7af87114b39712e6da79b0a377eb|4           |2018-01-18 00:00:00 |2018-01-18 21:46:59    |
|a548910a1c6147796b98fdf73dbeba33|5           |2018-03-10 00:00:00 |2018-03-11 03:05:13    |
|f9e4b658b201a9f2ecdecbb34bed034b|5           |2018-02-17 00:00:00 |2018-02-18 14:36:24    |
|658677c97b385a9be170737859d3511b|5           |2017-04-21 00:00:00 |2017-04-21 22:02:06    |
|8e6bfb81e283fa7e4f11123a3fb894f1|5           |2018-03-01 00:00:00 |2018-03-02 10:26:53    |
|b18dcdf73be66366873cd26c5724d1dc|1           |2018-04-13 00:00:00 |2018-04-16 00:39:37    |
|e48aa0d2dcec3a2e87348811bcfdf22b|5           |2017-07-16 00:00:00 |2017-07-18 19:30:34    |
|c31a859e34e3adac22f376954e19b39d|5           |2018-08-14 00:00:00 |20

In [49]:
reviews = reviews_raw.select(
    "order_id",
    col("review_score").cast("integer").alias("review_score")
)

reviews.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)



In [50]:

from pyspark.sql.functions import count, avg, min, max
review_summary = reviews.groupBy("order_id").agg(
    count("*").alias("review_count"),
    avg("review_score").alias("review_score_avg"),
    min("review_score").alias("review_score_min"),
    max("review_score").alias("review_score_max")
)

review_summary.show(5)

+--------------------+------------+----------------+----------------+----------------+
|            order_id|review_count|review_score_avg|review_score_min|review_score_max|
+--------------------+------------+----------------+----------------+----------------+
|e239d280236cdd3c4...|           1|             5.0|               5|               5|
|d451da9b109e1786f...|           1|             5.0|               5|               5|
|0de5dbc9f32267616...|           1|             4.0|               4|               4|
|251f0a3981c4a8cb8...|           1|             5.0|               5|               5|
|f63a31c3349b87273...|           2|             5.0|               5|               5|
+--------------------+------------+----------------+----------------+----------------+
only showing top 5 rows


In [51]:
order_items = order_items.join(
    review_summary,
    on="order_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 38


In [52]:
order_items = order_items.join(
    category_translation,
    on="product_category_name",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 39


In [53]:
order_summary = order_items.groupBy("order_id").agg(
    count("*").alias("order_item_count"),
    sum("item_total").alias("order_total_value"),
    sum("freight_value").alias("order_total_freight")
)

In [54]:
order_summary.show(5)

+--------------------+----------------+------------------+-------------------+
|            order_id|order_item_count| order_total_value|order_total_freight|
+--------------------+----------------+------------------+-------------------+
|014405982914c2cde...|               2| 78.42999999999999|               29.2|
|019886de8f385a39b...|               1|             188.4|               28.5|
|01a6ad782455876aa...|               1|             50.09|               15.1|
|01d907b3e209269e1...|               1|169.76000000000002|              17.77|
|028dc52e12ddda803...|               1|             61.72|              11.73|
+--------------------+----------------+------------------+-------------------+
only showing top 5 rows


In [55]:
order_items = order_items.join(
    order_summary,
    on="order_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 42


In [56]:
from pyspark.sql.functions import datediff

order_items = order_items.withColumn(
    "delivery_days",
    datediff("order_delivered_customer_date", 
            "order_purchase_timestamp")
)

In [57]:
order_items.select("order_id", "delivery_days").show(5)

+--------------------+-------------+
|            order_id|delivery_days|
+--------------------+-------------+
|00010242fe8c5a6d1...|            7|
|00018f77f2f0320c5...|           16|
|000229ec398224ef6...|            8|
|00024acbcdf0a6daa...|            6|
|00042b26cf59d7ce6...|           25|
+--------------------+-------------+
only showing top 5 rows


In [58]:
order_items = order_items.withColumn(
    "estimated_delivery_days",
    datediff(
        "order_estimated_delivery_date",
        "order_purchase_timestamp"
    )
)

order_items.select(
    "order_id",
    "estimated_delivery_days"
).show(5)

+--------------------+-----------------------+
|            order_id|estimated_delivery_days|
+--------------------+-----------------------+
|00010242fe8c5a6d1...|                     16|
|00018f77f2f0320c5...|                     19|
|000229ec398224ef6...|                     22|
|00024acbcdf0a6daa...|                     12|
|00042b26cf59d7ce6...|                     41|
+--------------------+-----------------------+
only showing top 5 rows


In [59]:
order_items = order_items.withColumn(
    "delivery_delay_days",
    datediff(
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

order_items.select(
    "order_id",
    "delivery_delay_days"
).show(10)

+--------------------+-------------------+
|            order_id|delivery_delay_days|
+--------------------+-------------------+
|00010242fe8c5a6d1...|                 -9|
|00018f77f2f0320c5...|                 -3|
|000229ec398224ef6...|                -14|
|00024acbcdf0a6daa...|                 -6|
|00042b26cf59d7ce6...|                -16|
|00048cc3ae777c65d...|                -15|
|00054e8431b9d7675...|                -17|
|000576fe39319847c...|                -16|
|0005a1a1728c9d785...|                  0|
|0005f50442cb953dc...|                -19|
+--------------------+-------------------+
only showing top 10 rows


In [60]:
from pyspark.sql.functions import when

order_items = order_items.withColumn(
    "is_delivered_late",
    when(col("delivery_delay_days") > 0, 1).otherwise(0)
)

order_items.select(
    "order_id",
    "delivery_delay_days",
    "is_delivered_late"
).show(10)

+--------------------+-------------------+-----------------+
|            order_id|delivery_delay_days|is_delivered_late|
+--------------------+-------------------+-----------------+
|00010242fe8c5a6d1...|                 -9|                0|
|00018f77f2f0320c5...|                 -3|                0|
|000229ec398224ef6...|                -14|                0|
|00024acbcdf0a6daa...|                 -6|                0|
|00042b26cf59d7ce6...|                -16|                0|
|00048cc3ae777c65d...|                -15|                0|
|00054e8431b9d7675...|                -17|                0|
|000576fe39319847c...|                -16|                0|
|0005a1a1728c9d785...|                  0|                0|
|0005f50442cb953dc...|                -19|                0|
+--------------------+-------------------+-----------------+
only showing top 10 rows


In [61]:
order_items = order_items.withColumn(
    "delivery_status",
    when(col("delivery_delay_days") > 0, "Late")
    .when(col("delivery_delay_days") == 0, "On Time")
    .otherwise("Early")
)

order_items.select(
    "order_id",
    "delivery_delay_days",
    "delivery_status"
).show(10)

+--------------------+-------------------+---------------+
|            order_id|delivery_delay_days|delivery_status|
+--------------------+-------------------+---------------+
|00010242fe8c5a6d1...|                 -9|          Early|
|00018f77f2f0320c5...|                 -3|          Early|
|000229ec398224ef6...|                -14|          Early|
|00024acbcdf0a6daa...|                 -6|          Early|
|00042b26cf59d7ce6...|                -16|          Early|
|00048cc3ae777c65d...|                -15|          Early|
|00054e8431b9d7675...|                -17|          Early|
|000576fe39319847c...|                -16|          Early|
|0005a1a1728c9d785...|                  0|        On Time|
|0005f50442cb953dc...|                -19|          Early|
+--------------------+-------------------+---------------+
only showing top 10 rows


In [62]:
order_items = order_items.withColumn(
    "delivery_status",
    when(col("delivery_delay_days").isNull(), "Not Delivered")
    .when(col("delivery_delay_days") > 0, "Late")
    .when(col("delivery_delay_days") == 0, "On Time")
    .otherwise("Early")
)

order_items.select(
    "order_id",
    "delivery_delay_days",
    "delivery_status"
).show(10)

+--------------------+-------------------+---------------+
|            order_id|delivery_delay_days|delivery_status|
+--------------------+-------------------+---------------+
|00010242fe8c5a6d1...|                 -9|          Early|
|00018f77f2f0320c5...|                 -3|          Early|
|000229ec398224ef6...|                -14|          Early|
|00024acbcdf0a6daa...|                 -6|          Early|
|00042b26cf59d7ce6...|                -16|          Early|
|00048cc3ae777c65d...|                -15|          Early|
|00054e8431b9d7675...|                -17|          Early|
|000576fe39319847c...|                -16|          Early|
|0005a1a1728c9d785...|                  0|        On Time|
|0005f50442cb953dc...|                -19|          Early|
+--------------------+-------------------+---------------+
only showing top 10 rows


In [63]:
customer_order_summary = order_items.groupBy("customer_id").agg(
    count("order_id").alias("customer_order_count")
)

customer_order_summary.show(10)

+--------------------+--------------------+
|         customer_id|customer_order_count|
+--------------------+--------------------+
|2e45292979b9b2700...|                   3|
|fc021f2dd664b4945...|                   1|
|03a7750fc7a7bfbd7...|                   1|
|456c1e01c8ed3b83a...|                   1|
|25ed0dc24f7c50b87...|                   2|
|6b994d4f438ab8322...|                   1|
|adf1ecaca5e192f13...|                   1|
|43d9a8df81f815755...|                   1|
|a7ab31829dc9a10f3...|                   6|
|98f405f18fec61e1a...|                   1|
+--------------------+--------------------+
only showing top 10 rows


In [64]:
from pyspark.sql.functions import countDistinct

customer_order_summary = order_items.groupBy("customer_id").agg(
    countDistinct("order_id").alias("customer_order_count")
)

customer_order_summary.show(10)

+--------------------+--------------------+
|         customer_id|customer_order_count|
+--------------------+--------------------+
|088935be5a63fd416...|                   1|
|1187acd131fe97cfc...|                   1|
|0b60138b038f460ba...|                   1|
|6b75e2df886b0fdde...|                   1|
|fdd48a32f403170cf...|                   1|
|42da09831872a4ecc...|                   1|
|c920352ccf11e554a...|                   1|
|6442504c76c94895e...|                   1|
|e139948a02a65f3d6...|                   1|
|bb36d8c5b34d858d1...|                   1|
+--------------------+--------------------+
only showing top 10 rows


In [65]:
customer_order_values = order_items.select(
    "customer_id",
    "order_id",
    "order_total_value"
).dropDuplicates(["customer_id", "order_id"])

customer_spend_summary = customer_order_values.groupBy("customer_id").agg(
    sum("order_total_value").alias("customer_total_spend")
)

customer_spend_summary.show(10)

+--------------------+--------------------+
|         customer_id|customer_total_spend|
+--------------------+--------------------+
|f82d1df095bf073a1...|              117.57|
|6442504c76c94895e...|               73.34|
|e139948a02a65f3d6...|              140.62|
|fb6f8e0cb2c65dfad...|              155.27|
|8baeca32aac79a831...|               59.87|
|0b60138b038f460ba...|               87.39|
|f39d2cea42fe92f3f...|               63.42|
|18143f5d1df138267...|                65.1|
|eb7b6d66f9d085148...|  45.099999999999994|
|fc021f2dd664b4945...|               55.11|
+--------------------+--------------------+
only showing top 10 rows


In [66]:
customer_summary = customer_order_summary.join(
    customer_spend_summary,
    on="customer_id",
    how="left"
)

customer_summary = customer_summary.withColumn(
    "customer_is_repeat",
    when(col("customer_order_count") > 1, 1).otherwise(0)
)

customer_summary.show(10)

+--------------------+--------------------+--------------------+------------------+
|         customer_id|customer_order_count|customer_total_spend|customer_is_repeat|
+--------------------+--------------------+--------------------+------------------+
|088935be5a63fd416...|                   1|               83.34|                 0|
|1187acd131fe97cfc...|                   1|               46.11|                 0|
|0b60138b038f460ba...|                   1|               87.39|                 0|
|6b75e2df886b0fdde...|                   1|              139.06|                 0|
|fdd48a32f403170cf...|                   1|               57.16|                 0|
|42da09831872a4ecc...|                   1|               88.83|                 0|
|c920352ccf11e554a...|                   1|               75.07|                 0|
|6442504c76c94895e...|                   1|               73.34|                 0|
|e139948a02a65f3d6...|                   1|              140.62|            

In [67]:
order_items = order_items.join(
    customer_summary,
    on="customer_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 50


In [68]:
seller_summary = order_items.groupBy("seller_id").agg(
    count("*").alias("seller_item_count")
)

seller_summary.show(10)

+--------------------+-----------------+
|           seller_id|seller_item_count|
+--------------------+-----------------+
|ff063b022a9a0aab9...|               96|
|8e6cc767478edae94...|              114|
|a49928bcdf77c55c6...|              106|
|da7039f29f90ce5b4...|               18|
|062ce95fa2ad4dfae...|               60|
|2009a095de2a2a416...|               12|
|0ea22c1cfbdc755f8...|              269|
|6eeed17989b0ae47c...|                3|
|e63e8bfa530fb1691...|               43|
|4d600e08ecbe08258...|                8|
+--------------------+-----------------+
only showing top 10 rows


In [69]:
seller_summary = seller_summary.join(
    order_items.groupBy("seller_id").agg(
        countDistinct("order_id").alias("seller_order_count")
    ),
    on="seller_id",
    how="left"
)

seller_summary.show(10)

+--------------------+-----------------+------------------+
|           seller_id|seller_item_count|seller_order_count|
+--------------------+-----------------+------------------+
|ff063b022a9a0aab9...|               96|                94|
|8e6cc767478edae94...|              114|               106|
|a49928bcdf77c55c6...|              106|                98|
|da7039f29f90ce5b4...|               18|                17|
|062ce95fa2ad4dfae...|               60|                55|
|2009a095de2a2a416...|               12|                 8|
|0ea22c1cfbdc755f8...|              269|               237|
|6eeed17989b0ae47c...|                3|                 3|
|e63e8bfa530fb1691...|               43|                42|
|4d600e08ecbe08258...|                8|                 8|
+--------------------+-----------------+------------------+
only showing top 10 rows


In [70]:
seller_sales = order_items.groupBy("seller_id").agg(
sum("item_total").alias("seller_total_sales")
)

seller_summary = seller_summary.join(
seller_sales,
on="seller_id",
how="left"
)

seller_summary.show(10)


+--------------------+-----------------+------------------+------------------+
|           seller_id|seller_item_count|seller_order_count|seller_total_sales|
+--------------------+-----------------+------------------+------------------+
|ff063b022a9a0aab9...|               96|                94|14061.270000000002|
|8e6cc767478edae94...|              114|               106|           9115.41|
|a49928bcdf77c55c6...|              106|                98|10574.250000000004|
|da7039f29f90ce5b4...|               18|                17|           1503.55|
|062ce95fa2ad4dfae...|               60|                55|           8721.44|
|2009a095de2a2a416...|               12|                 8|            582.43|
|0ea22c1cfbdc755f8...|              269|               237|15075.310000000003|
|6eeed17989b0ae47c...|                3|                 3|            676.01|
|e63e8bfa530fb1691...|               43|                42|1979.1699999999998|
|4d600e08ecbe08258...|                8|            

In [71]:
seller_summary = seller_summary.withColumn(
"seller_avg_order_value",
col("seller_total_sales") / col("seller_order_count")
)

seller_summary.show(10)


+--------------------+-----------------+------------------+------------------+----------------------+
|           seller_id|seller_item_count|seller_order_count|seller_total_sales|seller_avg_order_value|
+--------------------+-----------------+------------------+------------------+----------------------+
|ff063b022a9a0aab9...|               96|                94|14061.270000000002|     149.5879787234043|
|8e6cc767478edae94...|              114|               106|           9115.41|     85.99443396226415|
|a49928bcdf77c55c6...|              106|                98|10574.250000000004|    107.90051020408167|
|da7039f29f90ce5b4...|               18|                17|           1503.55|     88.44411764705882|
|062ce95fa2ad4dfae...|               60|                55|           8721.44|    158.57163636363637|
|2009a095de2a2a416...|               12|                 8|            582.43|              72.80375|
|0ea22c1cfbdc755f8...|              269|               237|15075.310000000003|    

In [72]:
order_items = order_items.join(
    seller_summary,
    on="seller_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 54


In [73]:
order_items = order_items.withColumn(
"customer_avg_order_value",
col("customer_total_spend") / col("customer_order_count")
)

order_items.select(
"customer_id",
"customer_order_count",
"customer_total_spend",
"customer_avg_order_value"
).show(10)


+--------------------+--------------------+--------------------+------------------------+
|         customer_id|customer_order_count|customer_total_spend|customer_avg_order_value|
+--------------------+--------------------+--------------------+------------------------+
|05b5f7bd9f0d0c31b...|                   1|               18.99|                   18.99|
|58dbd0b2d70206bf4...|                   1|  218.04000000000002|      218.04000000000002|
|7b078f45ddc9d7325...|                   1|               77.47|                   77.47|
|96fd44add323bdcc8...|                   1|              911.65|                  911.65|
|9ed5e522dd9dd85b4...|                   1|              880.75|                  880.75|
|a91d3b41b3f176b1e...|                   1|  234.85999999999999|      234.85999999999999|
|aae651df85accd493...|                   1|              129.86|                  129.86|
|c0d3ff03f73f41d3c...|                   1|               86.86|                   86.86|
|d6c4d0640

In [74]:
order_items = order_items.withColumn(
"freight_percentage",
(col("freight_value") / col("price")) * 100
)

order_items.select(
"price",
"freight_value",
"freight_percentage"
).show(10)


+------+-------------+------------------+
| price|freight_value|freight_percentage|
+------+-------------+------------------+
|  58.9|        13.29|22.563667232597624|
| 239.9|        19.93|  8.30762817840767|
| 199.0|        17.87| 8.979899497487438|
| 12.99|        12.79| 98.46035411855273|
| 199.9|        18.14| 9.074537268634316|
|  21.9|        12.69|57.945205479452056|
|  19.9|        11.85| 59.54773869346733|
| 810.0|        70.75| 8.734567901234568|
|145.95|        11.65| 7.982185680027408|
| 53.99|         11.4|21.115021300240784|
+------+-------------+------------------+
only showing top 10 rows


In [75]:
from pyspark.sql.functions import year

order_items = order_items.withColumn(
"order_year",
year("order_purchase_timestamp")
)

order_items.select(
"order_purchase_timestamp",
"order_year"
).show(10)


+------------------------+----------+
|order_purchase_timestamp|order_year|
+------------------------+----------+
|     2017-09-13 08:59:02|      2017|
|     2017-04-26 10:53:06|      2017|
|     2018-01-14 14:33:31|      2018|
|     2018-08-08 10:00:35|      2018|
|     2017-02-04 13:57:51|      2017|
|     2017-05-15 21:42:34|      2017|
|     2017-12-10 11:53:48|      2017|
|     2018-07-04 12:08:27|      2018|
|     2018-03-19 18:40:33|      2018|
|     2018-07-02 13:59:39|      2018|
+------------------------+----------+
only showing top 10 rows


In [76]:
from pyspark.sql.functions import month

order_items = order_items.withColumn(
"order_month",
month("order_purchase_timestamp")
)

order_items.select(
"order_purchase_timestamp",
"order_month"
).show(10)


+------------------------+-----------+
|order_purchase_timestamp|order_month|
+------------------------+-----------+
|     2017-09-13 08:59:02|          9|
|     2017-04-26 10:53:06|          4|
|     2018-01-14 14:33:31|          1|
|     2018-08-08 10:00:35|          8|
|     2017-02-04 13:57:51|          2|
|     2017-05-15 21:42:34|          5|
|     2017-12-10 11:53:48|         12|
|     2018-07-04 12:08:27|          7|
|     2018-03-19 18:40:33|          3|
|     2018-07-02 13:59:39|          7|
+------------------------+-----------+
only showing top 10 rows


In [77]:
final_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "item_total",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "total_payment_value",
    "payment_types",
    "max_payment_installments",
    "payment_count",
    "review_count",
    "review_score_avg",
    "review_score_min",
    "review_score_max",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "product_category_name_english",
    "order_item_count",
    "order_total_value",
    "order_total_freight",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "is_delivered_late",
    "delivery_status",
    "customer_order_count",
    "customer_total_spend",
    "customer_is_repeat",
    "seller_item_count",
    "seller_order_count",
    "seller_total_sales",
    "freight_percentage",
    "customer_avg_order_value",
    "seller_avg_order_value",
    "order_year",
    "order_month"
]

item_analysis_spark = order_items.select(final_columns)

print(item_analysis_spark.count(), len(item_analysis_spark.columns))

112650 58


In [78]:
item_analysis_spark.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+--------------------+------------+-------------------+-------------+------------------------+-------------+------------+----------------+----------------+----------------+--------------------+------------------------+--------------------+--------------+-----------------------------+----------------+------------------+-------------------+-------------+-----------------------+-------------------+-----------------+---------------+--------------------+--------------------+------------------+-----------------+----

In [79]:
print(item_analysis_spark.columns == final_columns)
print(len(item_analysis_spark.columns))


True
58


In [80]:
item_analysis_spark_sorted = item_analysis_spark.orderBy(
"order_id",
"order_item_id"
)

item_analysis_spark_sorted.select(
"order_id",
"order_item_id",
"item_total",
"customer_total_spend",
"seller_total_sales"
).show(10, truncate=False)


+--------------------------------+-------------+------------------+--------------------+------------------+
|order_id                        |order_item_id|item_total        |customer_total_spend|seller_total_sales|
+--------------------------------+-------------+------------------+--------------------+------------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |72.19             |72.19               |15183.640000000003|
|00018f77f2f0320c557190d7a144bdd3|1            |259.83            |259.83              |12071.999999999996|
|000229ec398224ef6ca0657da4fc703e|1            |216.87            |216.87              |3548.9500000000003|
|00024acbcdf0a6daa1e931b038114c75|1            |25.78             |25.78               |1331.87           |
|00042b26cf59d7ce69dfabb4e55b4fd9|1            |218.04000000000002|218.04000000000002  |4267.33           |
|00048cc3ae777c65dbb7d2a0634bc1ea|1            |34.589999999999996|34.589999999999996  |1593.82           |
|00054e8431b9d7675808bcb819f

In [81]:
from pyspark.sql.functions import col, sum, when

null_counts = item_analysis_spark.select([
sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
for c in item_analysis_spark.columns
])

null_counts.show(truncate=False)


+--------+-------------+----------+---------+-------------------+-----+-------------+----------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+-----------+------------+-------------------+-------------+------------------------+-------------+------------+----------------+----------------+----------------+------------------+------------------------+-------------+--------------+-----------------------------+----------------+-----------------+-------------------+-------------+-----------------------+-------------------+-----------------+---------------+--------------------+--------------------+------------------+-----------------+------------------+------------------+------------------+------------------

In [82]:
print("Total rows:", item_analysis_spark.count())

print(
"Unique order-item keys:",
item_analysis_spark.select(
"order_id",
"order_item_id"
).distinct().count()
)


Total rows: 112650
Unique order-item keys: 112650


In [83]:

item_analysis_spark.printSchema()



root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- item_total: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = t

In [84]:

item_analysis_spark.groupBy(
    "delivery_status"
).count().orderBy(
    "delivery_status"
).show()

item_analysis_spark.groupBy(
    "is_delivered_late"
).count().orderBy(
    "is_delivered_late"
).show()



+---------------+------+
|delivery_status| count|
+---------------+------+
|          Early|101481|
|           Late|  7265|
|  Not Delivered|  2454|
|        On Time|  1450|
+---------------+------+

+-----------------+------+
|is_delivered_late| count|
+-----------------+------+
|                0|105385|
|                1|  7265|
+-----------------+------+



In [85]:
output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\analytics_ready\spark\olist_item_analysis"

item_analysis_spark.write.mode("overwrite").parquet(output_path)

print("Parquet write successful!")

Parquet write successful!


In [86]:
parquet_check = spark.read.parquet(output_path)

print("Rows:", parquet_check.count())
print("Columns:", len(parquet_check.columns))
print("Columns match:", parquet_check.columns == final_columns)

Rows: 112650
Columns: 58
Columns match: True


In [87]:
single_parquet_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\analytics_ready\spark\olist_item_analysis_single.parquet"

item_analysis_spark.coalesce(1).write.mode("overwrite").parquet(single_parquet_path)

print("Single Parquet file written successfully!")

Single Parquet file written successfully!


In [88]:
import os

single_path = single_parquet_path

parquet_files = [
    f for f in os.listdir(single_path)
    if f.endswith(".parquet")
]

print("Parquet data files:", len(parquet_files))
print("File:", parquet_files[0] if parquet_files else "None")

Parquet data files: 1
File: part-00000-8105c75a-32fa-4a3f-a859-1f153791345b-c000.snappy.parquet
